# Forest Landscape Restoration (FLR) index: global comparison

Python port of the Earth Engine Code Editor script (AHP-weighted multi-criteria index), run for five study areas instead of the single `TG` asset:

**Togo, Cameroon, Benin, Madagascar, Ethiopia**

Steps, in the same order as the original script:

1. Study areas
2. Sentinel-2 cloud masking and spectral indices (NDVI, BSI, NDWI, NBAI)
3. Climate, terrain, soil, land cover, fire, crop, population, biomass, settlement, tree loss and groundwater layers
4. Simplified RUSLE erosion model
5. Min-max normalisation of every factor (per country)
6. AHP weights and the final `ForestLandRestoration` index
7. Maps
8. Reclassification into 4 classes and area per class
9. Comparison between countries
10. Exports to Google Drive

Requirements: `pip install earthengine-api geemap pandas matplotlib`

In [ ]:
import json
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Earth Engine project (taken from the asset path of the original script: projects/ee-typson09/...)
EE_PROJECT = "ee-typson09"

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

## Configuration

In [ ]:
COUNTRIES = ["Togo", "Cameroon", "Benin", "Madagascar", "Ethiopia"]

# Same periods as the original script
START, END = "2024-01-01", "2024-12-31"
FIRE_START, FIRE_END = "2024-01-01", "2024-05-01"

# Scales (metres)
# The Code Editor script used 30 m for both. Over whole countries that means billions of pixels of a
# one-year Sentinel-2 composite and the requests time out, so 1 km is the default here.
# Lower these (e.g. 250 or 100) for final numbers if the requests still complete.
MINMAX_SCALE = 1000  # scale at which each factor's min and max are taken
AREA_SCALE = 1000    # scale of the area-per-class calculation
EXPORT_SCALE_INDEX = 100    # original export scale of the final index
EXPORT_SCALE_FACTORS = 30   # original export scale of the normalised factors

# Whole-country statistics are computed tile by tile (and the tiles combined locally), because a
# single request over a large country exceeds Earth Engine's interactive time and memory limits.
TILE_DEG = 1.0       # tile size in degrees; make smaller if a tile still times out
TILE_WORKERS = 6     # parallel requests

OUT = Path("outputs")
(OUT / "data").mkdir(parents=True, exist_ok=True)
(OUT / "figures").mkdir(parents=True, exist_ok=True)

DRIVE_FOLDER = "EarthEngineExports"
RUN_EXPORTS = False  # set to True to start the Drive export tasks in the last section

## 1. Study areas

Country boundaries come from `USDOS/LSIB_SIMPLE/2017`. The original script used the asset `projects/ee-typson09/assets/TG`; to run that one as well, add it to `study_areas` as shown in the commented line.

In [ ]:
lsib = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")

study_areas = {
    name: lsib.filter(ee.Filter.eq("country_na", name)) for name in COUNTRIES
}
# study_areas["TG (original asset)"] = ee.FeatureCollection("projects/ee-typson09/assets/TG")

# Check that every country was found
for name, fc in study_areas.items():
    print(name, "features:", fc.size().getInfo())


def grid_tiles(study_area, deg=None):
    """Bounding box of the study area cut into deg x deg rectangles: [[x0, y0, x1, y1], ...]."""
    deg = deg or TILE_DEG
    xs, ys = zip(*study_area.geometry().bounds().coordinates().getInfo()[0])
    x_edges = np.arange(min(xs), max(xs), deg)
    y_edges = np.arange(min(ys), max(ys), deg)
    return [[float(x), float(y), float(min(x + deg, max(xs))), float(min(y + deg, max(ys)))]
            for x in x_edges for y in y_edges]


def run_parallel(fn, items, workers=None, retries=4, label=""):
    """Run fn(item) in threads, retrying failed requests. Returns results in order (None if failed)."""
    def safe(item):
        err = ""
        for attempt in range(retries):
            try:
                return fn(item)
            except Exception as e:
                err = str(e)[:100]
                # "429 Too Many Requests": Earth Engine is throttling us, so wait much longer
                wait = 30 if "429" in err or "Too Many" in err else 5
                time.sleep(wait * (attempt + 1) + 5 * np.random.rand())
        print(f"  {label} failed after {retries} attempts: {err}")
        return None

    with ThreadPoolExecutor(workers or TILE_WORKERS) as pool:
        return list(pool.map(safe, items))

## 2. Sentinel-2 masking and spectral indices

SCL classes: 0 NO_DATA, 1 SATURATED_OR_DEFECTIVE, 2 DARK_FEATURES, 3 CLOUD_SHADOW, 4 VEGETATION, 5 NOT_VEGETATED, 6 WATER, 7 UNCLASSIFIED, 8 CLOUD_MEDIUM_PROBABILITY, 9 CLOUD_HIGH_PROBABILITY, 10 THIN_CIRRUS, 11 SNOW_ICE.

Masked out: cloud shadow (3), saturated/defective (1), clouds, cirrus and snow (>= 8).

In [ ]:
def mask_s2_sr(image):
    """Cloud and shadow masking with the SCL band."""
    scl = image.select("SCL")
    mask = scl.eq(3).Or(scl.eq(1)).Or(scl.gte(8))
    return image.updateMask(mask.Not()).copyProperties(
        image, ["system:time_start", "CLOUDY_PIXEL_PERCENTAGE"]
    )


def s2_ndvi(image):
    return image.normalizedDifference(["B8", "B4"]).rename("ndvi")


def s2_bsi(image):
    # BSI = ((B11 + B4) - (B8 + B2)) / ((B11 + B4) + (B8 + B2))
    return image.expression(
        "((B11 + B4) - (B8 + B2)) / ((B11 + B4) + (B8 + B2))",
        {
            "B2": image.select("B2"),
            "B4": image.select("B4"),
            "B8": image.select("B8"),
            "B11": image.select("B11"),
        },
    ).rename("bsi")


def s2_ndwi(image):
    # NDWI as defined in the original script: (B3 - B11) / (B3 + B11)
    return image.normalizedDifference(["B3", "B11"]).rename("ndwi")


def s2_nbai(image):
    # NBAI as defined in the original script: (B12 - B8) / (B12 + B8)
    return image.normalizedDifference(["B12", "B8"]).rename("nbai")

## 3 - 4. Input layers and RUSLE erosion

`build_factors(study_area)` returns every layer of the original script for one study area, each as a single-band image named after its key.

In [ ]:
def build_factors(study_area):
    """All raw (non-normalised) layers of the original script for one study area."""
    # ---- Sentinel-2 ----
    s2 = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(study_area)
        .filterDate(START, END)
    )
    s2_masked = s2.map(mask_s2_sr)

    ndvi = s2_masked.map(s2_ndvi).median().clip(study_area)
    bsi = s2_masked.map(s2_bsi).median().clip(study_area)
    ndwi = s2_masked.map(s2_ndwi).median().clip(study_area)
    nbai = s2_masked.map(s2_nbai).median().clip(study_area)

    # ---- CHIRPS precipitation (annual sum) ----
    precipitation = (
        ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
        .filterBounds(study_area)
        .filterDate(START, END)
        .select("precipitation")
        .sum()
        .rename("precipitation")
        .clip(study_area)
    )

    # ---- Elevation and slope (SRTM) ----
    dem = ee.Image("USGS/SRTMGL1_003").clip(study_area)
    elevation = dem.select("elevation").rename("elevation")
    slope = ee.Terrain.slope(dem).rename("slope")

    # ---- Soil organic carbon (iSDAsoil, 20-50 cm) ----
    soc = (
        ee.Image("ISDASOIL/Africa/v1/carbon_organic")
        .select("mean_20_50")
        .divide(10)
        .rename("soc")
        .clip(study_area)
    )

    # ---- Land use / land cover (ESA WorldCover v100) ----
    lulc = (
        ee.ImageCollection("ESA/WorldCover/v100")
        .first()
        .select("Map")
        .rename("lulc")
        .clip(study_area)
    )

    # ---- TerraClimate: soil moisture and max temperature ----
    terraclimate = (
        ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE")
        .filterBounds(study_area)
        .filterDate(START, END)
    )
    soil_moisture = (
        terraclimate.select("soil").mean().rename("soilMoisture").clip(study_area)
    )
    max_temperature = (
        terraclimate.select("tmmx")
        .map(lambda img: img.multiply(0.1).copyProperties(img, ["system:time_start"]))
        .mean()
        .rename("maxTemperature")
        .clip(study_area)
    )

    # ---- Drainage (MERIT Hydro, upstream contributing area, km2) ----
    drainage = (
        ee.Image("MERIT/Hydro/v1_0_1").select("upa").rename("drainage").clip(study_area)
    )

    # ---- Soil texture and pH (iSDAsoil, 20-50 cm) ----
    soil_texture = (
        ee.Image("ISDASOIL/Africa/v1/texture_class")
        .select("texture_20_50")
        .rename("soilTexture")
        .clip(study_area)
    )
    soil_ph = (
        ee.Image("ISDASOIL/Africa/v1/ph")
        .select("mean_20_50")
        .divide(10)
        .rename("soilPh")
        .clip(study_area)
    )

    # ---- MODIS fire mask ----
    fire_mask = (
        ee.ImageCollection("MODIS/061/MOD14A1")
        .filter(ee.Filter.date(FIRE_START, FIRE_END))
        .max()
        .select("FireMask")
        .rename("fireMask")
        .clip(study_area)
    )

    # ---- Crop mask (GFSAD1000) ----
    crop_mask = (
        ee.Image("USGS/GFSAD1000_V1")
        .select("landcover")
        .rename("cropMask")
        .clip(study_area)
    )

    # ---- Population density (GPWv4.11) ----
    population_density = (
        ee.ImageCollection("CIESIN/GPWv411/GPW_UNWPP-Adjusted_Population_Density")
        .first()
        .select("unwpp-adjusted_population_density")
        .rename("population_density")
        .clip(study_area)
    )

    # ---- LandScan population count (latest image; visualisation only, not in the index) ----
    landscan_population = (
        ee.ImageCollection("projects/sat-io/open-datasets/ORNL/LANDSCAN_GLOBAL")
        .sort("system:time_start", False)
        .first()
        .select("b1")
        .rename("landscan_population")
        .clip(study_area)
    )

    # ---- Biomass carbon density (WCMC 2010) ----
    biomas_c = (
        ee.Image("WCMC/biomass_carbon_density/v1_0/2010")
        .select("carbon_tonnes_per_ha")
        .rename("biomasC")
        .clip(study_area)
    )

    # ---- Settlement (GHSL built-up characteristics 2018) ----
    settlement = (
        ee.Image("JRC/GHSL/P2023A/GHS_BUILT_C/2018")
        .select("built_characteristics")
        .rename("settlement")
        .clip(study_area)
    )

    # ---- Tree loss year (Hansen GFC 2024 v1.12); 0 = no loss ----
    # lossyear is masked wherever no loss occurred. Left as it is, the index would only exist on
    # loss pixels. No-loss pixels are set to 0; multiplying by the mask and then resetting the mask
    # keeps this correct at coarser scales too (value = loss year averaged over the whole cell).
    lossyear = ee.Image("UMD/hansen/global_forest_change_2024_v1_12").select("lossyear")
    treeloss = (
        lossyear.unmask(0)
        .multiply(lossyear.mask())
        .mask(ee.Image.constant(1))
        .rename("treeloss")
        .clip(study_area)
    )

    # ---- Underground water (GRACE, liquid water equivalent thickness, CSR) ----
    underground_water = (
        ee.Image("NASA/GRACE/MASS_GRIDS_V04/LAND/20170501_20170522")
        .select("lwe_thickness_csr")
        .rename("undergroundWater")
        .clip(study_area)
    )

    # ---- RUSLE erosion model: A = R * K * LS * C * P ----
    # LS: simplified linear approximation from slope (degrees)
    LS = slope.multiply(0.017).rename("LS_Factor")

    # R: mean daily CHIRPS rainfall (factor of 1 applied, as in the original)
    rainfall = (
        ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
        .filterDate(START, END)
        .mean()
        .rename("R_Factor")
        .clip(study_area)
    )
    R = rainfall.multiply(1)

    # K: OpenLandMap organic carbon used as a proxy
    K = (
        ee.Image("OpenLandMap/SOL/SOL_ORGANIC-CARBON_USDA-6A1C_M/v02")
        .select("b0")
        .divide(10)
        .rename("K_Factor")
        .clip(study_area)
    )

    # C: 1 - NDVI, from scenes with < 20 % cloudy pixels
    s2_c = s2_masked.filterDate(START, END).filter(
        ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20)
    )
    ndvi_c = s2_c.map(s2_ndvi).median().rename("ndvi_c_factor").clip(study_area)
    C = ndvi_c.multiply(-1).add(1).rename("C_Factor")

    # P: assumed 1
    P = ee.Image.constant(1).rename("P_Factor").clip(study_area)

    erosion = R.multiply(K).multiply(LS).multiply(C).multiply(P).rename("erosion")

    # Factors that enter the index (same keys and order as imagesToNormalize)
    factors = {
        "ndvi": ndvi,
        "precipitation": precipitation,
        "slope": slope,
        "soc": soc,
        "soilTexture": soil_texture,
        "soilMoisture": soil_moisture,
        "maxTemperature": max_temperature,
        "drainage": drainage,
        "erosion": erosion,
        "bsi": bsi,
        "ndwi": ndwi,
        "nbai": nbai,
        "fireMask": fire_mask,
        "cropMask": crop_mask,
        "population_density": population_density,
        "biomasC": biomas_c,
        "settlement": settlement,
        "treeloss": treeloss,
        "undergroundWater": underground_water,
        "lulc": lulc,
    }
    # Layers computed in the original script but not used in the index
    extras = {
        "elevation": elevation,
        "soilPh": soil_ph,
        "landscan_population": landscan_population,
        "LS_Factor": LS,
        "R_Factor": R,
        "K_Factor": K,
        "C_Factor": C,
        "P_Factor": P,
    }
    return factors, extras

## 5. Normalisation

Min-max scaling `(value - min) / (max - min)`, with min and max computed **per study area** at `MINMAX_SCALE`. They are computed tile by tile, combined, and cached in `outputs/data/minmax_<country>_<scale>m.json`; delete that file to recompute. LULC is the exception, as in the original: the WorldCover class code is divided by 100.

In [ ]:
def country_min_max(name, factors, study_area):
    """Min and max of every factor over the study area, tile by tile, cached as JSON."""
    path = OUT / "data" / f"minmax_{name.replace(' ', '_')}_{MINMAX_SCALE}m.json"
    if path.exists():
        return json.loads(path.read_text())

    all_factors = ee.Image(list(factors.values()))   # each band is already clipped to the study area

    def tile_stats(t):
        return all_factors.reduceRegion(
            reducer=ee.Reducer.minMax(), geometry=ee.Geometry.Rectangle(t, None, False),
            scale=MINMAX_SCALE, maxPixels=1e10, tileScale=4,
        ).getInfo()

    tiles = grid_tiles(study_area)
    stats = run_parallel(tile_stats, tiles, label=f"{name} min/max tile")
    failed = sum(s is None for s in stats)
    if failed:
        raise RuntimeError(f"{name}: {failed} of {len(tiles)} tiles failed; lower TILE_DEG and re-run.")

    min_max = {}
    for key in factors:
        lo = [s[key + "_min"] for s in stats if s.get(key + "_min") is not None]
        hi = [s[key + "_max"] for s in stats if s.get(key + "_max") is not None]
        if not lo:
            raise RuntimeError(f"{name}: factor '{key}' has no data in this study area.")
        min_max[key + "_min"], min_max[key + "_max"] = min(lo), max(hi)
    path.write_text(json.dumps(min_max, indent=1))
    return min_max


def normalize_factors(name, factors, study_area):
    """Min-max normalise every factor. Returns {key: image named '<key>_norm'}."""
    min_max = country_min_max(name, factors, study_area)

    def normalize(band):
        mn, mx = min_max[band + "_min"], min_max[band + "_max"]
        rng = (mx - mn) or 1          # avoid division by zero for constant images
        return factors[band].subtract(mn).divide(rng).rename(band + "_norm")

    normalized = {key: normalize(key) for key in factors}

    # LULC: categorical simplification, class code / 100
    normalized["lulc"] = factors["lulc"].divide(100).rename("lulc_norm")
    return normalized, min_max

## 6. AHP weights and the index

Weights are copied from the original script (derive your own with the AHP calculator at https://bpmsg.com/ahp/ahp-calc.php).

**`soilTexture` corrected:** the Code Editor script has `0.89`, which makes the weights sum to 1.803. It is set to `0.089` here, so the weights sum to 1.002 and the index stays within 0-1. Results therefore differ from the Code Editor output until the same correction is made there.

In [ ]:
SOIL_TEXTURE_WEIGHT = 0.089  # 0.89 in the Code Editor script (typo); corrected

weights = {
    "ndvi": 0.139,
    "precipitation": 0.138,
    "slope": 0.077,
    "soilTexture": SOIL_TEXTURE_WEIGHT,
    "lulc": 0.121,
    "soilMoisture": 0.091,
    "maxTemperature": 0.031,
    "drainage": 0.058,
    "erosion": 0.028,
    "bsi": 0.019,
    "ndwi": 0.042,
    "soc": 0.019,
    "nbai": 0.018,
    "fireMask": 0.015,
    "cropMask": 0.031,
    "population_density": 0.012,
    "biomasC": 0.023,
    "settlement": 0.011,
    "treeloss": 0.014,
    "undergroundWater": 0.026,
}

print("Number of factors:", len(weights))
print("Sum of weights:", round(sum(weights.values()), 3))
pd.Series(weights, name="weight").sort_values(ascending=False).to_frame()

In [ ]:
def weighted_index(normalized):
    """ForestLandRestoration = sum(normalised factor * weight)."""
    index = None
    for key, w in weights.items():
        term = normalized[key].multiply(w)
        index = term if index is None else index.add(term)
    return index.rename("ForestLandRestoration")


def classify_index(index):
    """4 classes: 1 Very Low (<=0.25), 2 Low (0.25-0.50), 3 Moderate (0.50-0.75), 4 High (>0.75)."""
    return (
        index.where(index.gt(0.75), 4)
        .where(index.gt(0.50).And(index.lte(0.75)), 3)
        .where(index.gt(0.25).And(index.lte(0.50)), 2)
        .where(index.lte(0.25), 1)
        .rename("ForestLandRestoration_Class")
        .toUint8()
    )

### Build everything for the five countries

The first run computes each country's min/max (several minutes per country, longest for Ethiopia); later runs read them from the cache.

In [ ]:
results = {}
for name, study_area in study_areas.items():
    factors, extras = build_factors(study_area)
    normalized, min_max = normalize_factors(name, factors, study_area)
    print("ready:", name)
    index = weighted_index(normalized)
    results[name] = {
        "study_area": study_area,
        "factors": factors,
        "extras": extras,
        "normalized": normalized,
        "min_max": min_max,
        "index": index,
        "classified": classify_index(index),
    }

list(results)

## 7. Maps

### Visualisation parameters (same as the original script)

In [ ]:
flr_vis = {
    "min": 0,
    "max": 1,
    "palette": ["#313695", "#4575b4", "#74add1", "#abd9e9", "#e0f3f8", "#ffffbf",
                "#fee090", "#fdae61", "#f46d43", "#d73027", "#a50026"],
}

class_vis = {"min": 1, "max": 4, "palette": ["#33a02c", "#b2df8a", "#fb9a99", "#e31a1c"]}
class_legend = {
    "1: Very Low": "#33a02c",
    "2: Low": "#b2df8a",
    "3: Moderate": "#fb9a99",
    "4: High": "#e31a1c",
}

fire_mask_vis = {
    "min": 0,
    "max": 9,
    # 0-6: no fire; 7: low; 8: nominal; 9: high confidence
    "palette": ["black"] * 7 + ["red", "orange", "yellow"],
}
crop_mask_vis = {"min": 0.0, "max": 5.0,
                 "palette": ["black", "orange", "brown", "02a50f", "green", "yellow"]}
population_density_vis = {"min": 0.0, "max": 1000.0,
                          "palette": ["ffffe7", "FFc869", "ffac1d", "e17735", "f2552c", "9f0c21"]}
tree_loss_vis = {"bands": ["treeloss"], "min": 0, "max": 24, "palette": ["black", "green"]}
groundwater_vis = {"palette": ["001137", "01abab", "e7eb05", "620500"],
                   "min": -0.7845402964290882, "max": 0.5693220260300134}

# Optional factor layers: key -> (vis params, layer name)
factor_vis = {
    "lulc": ({"min": 0, "max": 90, "palette": ["006633", "E6B8AF", "337029", "D6FFD6", "54D9AF",
                                                 "D4A786", "774230", "D7CDCC", "F7E1E1", "E1E1E1"]},
             "Land Use Land Cover"),
    "soilMoisture": ({"min": 0, "max": 0.6}, "Soil Moisture (mm)"),
    "maxTemperature": ({"min": 0, "max": 323}, "Max Temperature (C)"),
    "drainage": ({"min": 0, "max": 1000}, "Drainage (Upstream Area km^2)"),
    "soilTexture": ({"min": 1, "max": 12, "palette": ["fef0d9", "fdd49e", "fdb863", "fb8d3c",
                                                        "e6550d", "a63603"]}, "Soil Texture Class"),
    "erosion": ({"min": 0, "max": 5000}, "Erosion (RUSLE, tons/ha/yr)"),
    "ndvi": ({"min": -1, "max": 1, "palette": ["red", "white", "green"]}, "NDVI"),
    "precipitation": ({"min": 0, "max": 2000}, "Precipitation (mm/yr)"),
    "bsi": ({"min": -1, "max": 1, "palette": ["blue", "white", "brown"]}, "Bare Soil Index (BSI)"),
    "ndwi": ({"min": -1, "max": 1, "palette": ["brown", "white", "blue"]},
             "Normalized Difference Water Index (NDWI)"),
    "soc": ({"min": 1, "max": 180, "palette": ["brown", "white", "blue"]},
            "Soil organic carbon (soc)"),
    "nbai": ({"min": -1, "max": 1, "palette": ["green", "white", "purple"]},
             "Normalized Build-up Area Index (NBAI)"),
    "fireMask": (fire_mask_vis, "Fire Mask (MODIS)"),
    "cropMask": (crop_mask_vis, "Crop Mask"),
    "population_density": (population_density_vis, "Population Density"),
    "biomasC": ({"min": 1, "max": 180, "palette": ["d9f0a3", "addd8e", "78c679", "41ab5d",
                                                     "238443", "005a32"]},
                "Biomass Carbon Density (tonnes/ha)"),
    "settlement": ({}, "Settlement Characteristics (2018)"),
    "treeloss": (tree_loss_vis, "Tree Loss 2024"),
    "undergroundWater": (groundwater_vis, "Underground Water Thickness CSR"),
    "slope": ({"min": 0, "max": 60, "palette": ["lightgreen", "darkgreen", "orange", "red",
                                                  "darkred"]}, "Slope (degrees)"),
}

# SLD style for the LandScan population count
pop_style = (
    '<RasterSymbolizer><ColorMap type="intervals" extended="false">'
    '<ColorMapEntry color="#CCCCCC" quantity="0" label="No Data"/>'
    '<ColorMapEntry color="#FFFFBE" quantity="5"/>'
    '<ColorMapEntry color="#FEFF73" quantity="25"/>'
    '<ColorMapEntry color="#FEFF2C" quantity="50"/>'
    '<ColorMapEntry color="#FFAA27" quantity="100"/>'
    '<ColorMapEntry color="#FF6625" quantity="500"/>'
    '<ColorMapEntry color="#FF0023" quantity="2500"/>'
    '<ColorMapEntry color="#CC001A" quantity="5000"/>'
    '<ColorMapEntry color="#730009" quantity="185000"/>'
    "</ColorMap></RasterSymbolizer>"
)
pop_legend = {
    "0": "#CCCCCC", "1-5": "#FFFFBE", "6-25": "#FEFF73", "26-50": "#FEFF2C",
    "51-100": "#FFAA27", "101-500": "#FF6625", "501-2500": "#FF0023",
    "2501-5000": "#CC001A", "5001-185000": "#730009",
}


def outline(study_area):
    return ee.Image().byte().paint(featureCollection=study_area, color=1, width=3)

### FLR index for all five countries

Tiles are computed on the fly from a full year of Sentinel-2, so the first draw of each country can take a while, especially for Ethiopia and Madagascar.

In [ ]:
m_all = geemap.Map(center=[3, 25], zoom=4)
for name, r in results.items():
    m_all.addLayer(r["index"].clip(r["study_area"]), flr_vis, f"FLR index - {name}")
    m_all.addLayer(r["classified"].clip(r["study_area"]), class_vis, f"FLR classes - {name}", False)
    m_all.addLayer(outline(r["study_area"]), {"palette": ["red"]}, f"Outline - {name}")
m_all.add_colorbar(flr_vis, label="ForestLandRestoration index")
m_all.add_legend(title="ForestLandRestoration Index", legend_dict=class_legend)
m_all

### All layers for one country

Same layer stack as the original script. Change `COUNTRY` to look at another study area.

In [ ]:
COUNTRY = "Togo"

r = results[COUNTRY]
m = geemap.Map()
m.centerObject(r["study_area"], 7)
m.addLayer(outline(r["study_area"]), {"palette": ["red"]}, "Study Area Outline")
m.addLayer(r["index"].clip(r["study_area"]), flr_vis, "FRL Susceptibility (AHP)")
for key, (vis, label) in factor_vis.items():
    m.addLayer(r["factors"][key], vis, label, False)
m.addLayer(r["extras"]["landscan_population"].sldStyle(pop_style), {},
           "Population (LandScan, latest)", False)
m.addLayer(r["classified"].clip(r["study_area"]), class_vis,
           "ForestLandRestoration (4 Classes)", False)
m.add_legend(title="ForestLandRestoration Index", legend_dict=class_legend)
m

## 8. Area per class

Area in hectares of each of the four classes, per country. The original script made one `reduceRegion` per class; here a single grouped reducer returns all four at once, and it runs tile by tile so that large countries stay within Earth Engine's limits.

A tile that fails is automatically cut into four smaller ones and retried. The result is cached in `outputs/data/area_<country>_<scale>m.json`; delete that file to recompute.

In [ ]:
CLASS_NAMES = {1: "Very Low", 2: "Low", 3: "Moderate", 4: "High"}


def class_areas(name, r):
    """Hectares per class for one result entry, summed over tiles and cached as JSON."""
    path = OUT / "data" / f"area_{name.replace(' ', '_')}_{AREA_SCALE}m.json"
    if path.exists():
        return json.loads(path.read_text())
    classified = r["classified"]
    stack = ee.Image.pixelArea().updateMask(classified.mask()).addBands(classified)

    def tile_area(t):
        return stack.reduceRegion(
            reducer=ee.Reducer.sum().group(groupField=1, groupName="class"),
            geometry=ee.Geometry.Rectangle(t, None, False),
            scale=AREA_SCALE, maxPixels=1e10, tileScale=4,
        ).get("groups").getInfo()

    # A tile that fails (usually "User memory limit exceeded") is cut into four and tried again.
    pending, groups = grid_tiles(r["study_area"]), []
    for depth in range(3):
        done = run_parallel(tile_area, pending, retries=2, label=f"{name} area tile (will be split)")
        groups += [g for g in done if g is not None]
        failed = [t for t, g in zip(pending, done) if g is None]
        pending = [q for x0, y0, x1, y1 in failed
                   for q in ([x0, y0, (x0 + x1) / 2, (y0 + y1) / 2], [(x0 + x1) / 2, y0, x1, (y0 + y1) / 2],
                             [x0, (y0 + y1) / 2, (x0 + x1) / 2, y1], [(x0 + x1) / 2, (y0 + y1) / 2, x1, y1])]
        if not pending:
            break
    if pending:
        raise RuntimeError(f"{len(failed)} tiles still fail after splitting; raise AREA_SCALE and re-run.")
    areas = {label: 0.0 for label in CLASS_NAMES.values()}
    for tile_groups in groups:
        for g in tile_groups:
            areas[CLASS_NAMES[int(g["class"])]] += g["sum"] / 10000  # m2 -> ha
    path.write_text(json.dumps(areas, indent=1))
    return areas


area_rows, errors = {}, {}
for name, r in results.items():
    try:
        area_rows[name] = class_areas(name, r)
        print("done:", name)
    except Exception as e:
        errors[name] = str(e)
        print("FAILED:", name, "-", e)

area_ha = pd.DataFrame(area_rows).T.reindex(columns=list(CLASS_NAMES.values()))
area_ha.index.name = "country"
area_ha.round(0)

## 9. Comparison between countries

Because every factor is min-max normalised within its own country, the index is a *relative* ranking inside each country. Shares of area per class are comparable in that sense; absolute index values are not on a common scale across countries.

In [ ]:
area_pct = area_ha.div(area_ha.sum(axis=1), axis=0) * 100
area_pct.round(1)

In [ ]:
class_colors = ["#33a02c", "#b2df8a", "#fb9a99", "#e31a1c"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

area_pct.plot(kind="barh", stacked=True, color=class_colors, ax=axes[0], width=0.7)
axes[0].set_xlabel("Share of country area (%)")
axes[0].set_ylabel("")
axes[0].set_title("FLR classes: share of area")
axes[0].set_xlim(0, 100)
axes[0].invert_yaxis()
axes[0].legend(title="Class", bbox_to_anchor=(0.5, -0.15), loc="upper center", ncol=4, frameon=False)

(area_ha / 1e6).plot(kind="barh", stacked=True, color=class_colors, ax=axes[1], width=0.7, legend=False)
axes[1].set_xlabel("Area (million ha)")
axes[1].set_ylabel("")
axes[1].set_title("FLR classes: absolute area")
axes[1].invert_yaxis()

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Save the comparison tables
area_ha.round(2).to_csv(OUT / "data" / "flr_class_area_ha.csv")
area_pct.round(3).to_csv(OUT / "data" / "flr_class_area_pct.csv")

## 10. Exports to Google Drive

Same exports as the original script, once per country: the final index at 100 m and each normalised factor at 30 m (20 tasks per country, 100 in total). The classified map export was commented out in the original and is controlled by `EXPORT_CLASSIFIED` here.

Nothing is started unless `RUN_EXPORTS = True` in the configuration cell. Country-wide 30 m exports are large (many GeoTIFF tiles per layer, particularly for Ethiopia, Madagascar and Cameroon); raise `EXPORT_SCALE_FACTORS` if you do not need 30 m.

In [ ]:
EXPORT_CLASSIFIED = False

# key -> (task description, file name prefix) as in the original script
factor_exports = {
    "soc": ("soc_Map", "soc"),
    "ndvi": ("NDVI_Map", "NDVI"),
    "precipitation": ("Precipitation_Map", "Precipitation"),
    "slope": ("Slope_Map", "Slope"),
    "soilTexture": ("SoilTexture_Map", "SoilTexture"),
    "lulc": ("LULC_Map", "LULC"),
    "soilMoisture": ("SoilMoisture_Map", "SoilMoisture"),
    "maxTemperature": ("MaxTemperature_Map", "MaxTemperature"),
    "drainage": ("Drainage_Map", "Drainage"),
    "erosion": ("Erosion_Map", "Erosion"),
    "bsi": ("BSI_Map", "BSI"),
    "ndwi": ("NDWI_Map", "NDWI"),
    "nbai": ("NBAI_Map", "NBAI"),
    "fireMask": ("FireMask_Map", "FireMask"),
    "cropMask": ("CropMask_Map", "CropMask"),
    "population_density": ("PopulationDensity_Map", "PopulationDensity"),
    "biomasC": ("BiomasC_Map", "BiomasC"),
    "settlement": ("Settlement_Map", "Settlement"),
    "treeloss": ("Treeloss_Map", "Treeloss"),
    "undergroundWater": ("UndergroundWater_Map", "UndergroundWater"),
}


def export_country(name, r, start=False):
    """Create (and optionally start) the Drive export tasks for one country."""
    tag = name.replace(" ", "_").replace("(", "").replace(")", "")
    region = r["study_area"].geometry()
    tasks = []

    def make(image, description, prefix, scale):
        tasks.append(
            ee.batch.Export.image.toDrive(
                image=image.clip(r["study_area"]),
                description=f"{tag}_{description}",
                folder=DRIVE_FOLDER,
                fileNamePrefix=f"{tag}_{prefix}",
                region=region,
                scale=scale,
                maxPixels=1e13,
                fileFormat="GeoTIFF",
            )
        )

    make(r["index"], "ForestLandRestoration_AHP", "ForestLandRestoration_Map", EXPORT_SCALE_INDEX)
    if EXPORT_CLASSIFIED:
        make(r["classified"], "ForestLandRestoration_Classified_Map",
             "ForestLandRestoration_Classified_Map", EXPORT_SCALE_INDEX)
    for key, (description, prefix) in factor_exports.items():
        make(r["normalized"][key], description, prefix, EXPORT_SCALE_FACTORS)

    if start:
        for t in tasks:
            t.start()
    return tasks


all_tasks = {}
if RUN_EXPORTS:
    for name, r in results.items():
        all_tasks[name] = export_country(name, r, start=True)
        print(f"{name}: started {len(all_tasks[name])} export tasks")
else:
    print("RUN_EXPORTS is False: no export task started.")

In [ ]:
# Status of the export tasks started above
if all_tasks:
    status = [
        {"country": name, "task": t.status()["description"], "state": t.status()["state"]}
        for name, tasks in all_tasks.items() for t in tasks
    ]
    display(pd.DataFrame(status))

---
# Part B. Variable atlas and complementary analyses

Everything below works from a **random point sample** of each country (all raw layers, all normalised layers, the index and a few independent layers), downloaded once and cached as CSV in `outputs/data/`. The statistical analyses then run locally in pandas/numpy, so they are fast and reproducible and do not hit Earth Engine limits.

| Section | Analysis | Purpose |
|---|---|---|
| 11 | Point sample | Common data table for all analyses |
| 12 | Variable atlas per country, and the FLR index for all countries together | The 20 input variables; the main result figure |
| 13 | Descriptive statistics | Table 1 of a paper |
| 14 | Collinearity (Spearman, VIF) | Are factors redundant / double counted? |
| 15 | PCA-derived weights vs AHP | Is the expert weighting supported by the data? |
| 16 | Monte Carlo weight uncertainty | How stable are the classes? |
| 17 | One-at-a-time sensitivity | Which weights drive the result? |
| 18 | Factor contributions | What drives the index in each country? |
| 19 | Pooled normalisation | Makes countries comparable on one scale |
| 20 | Validation against independent data | Does the index relate to observed tree-cover gain and greening? |
| 21 | Stratification | Biome, protection status, admin-1 ranking |
| 22 | Direction-aware index (degradation / restoration need) | Signed factors, scored classes, comparable scale, quartile classes |
| 23 | What is still needed for a top-tier journal | Roadmap |

In [ ]:
import io
import hashlib

import requests
from PIL import Image
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Patch

N_SAMPLES = 3000      # points per country
SAMPLE_CHUNK = 250    # points per Earth Engine request
SAMPLE_SCALE = 100    # metres
N_MC = 1000           # Monte Carlo draws of the weight vector
MC_SIGMA = 0.25       # log-normal spread of the weight perturbation (about +/-25 %)
THUMB_PX = 700        # longest side of each atlas panel, in pixels
(OUT / "cache").mkdir(parents=True, exist_ok=True)   # downloaded map pictures, reused on re-runs

# Figure style close to journal requirements (sans-serif, small type, editable text in PDF)
plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 7, "axes.titlesize": 7, "axes.labelsize": 7,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 6,
    "axes.linewidth": 0.5, "pdf.fonttype": 42, "savefig.dpi": 600, "figure.dpi": 120,
})


def spearman(a, b):
    """Spearman rank correlation (no scipy needed)."""
    d = pd.concat([pd.Series(np.asarray(a, float)), pd.Series(np.asarray(b, float))], axis=1).dropna()
    return d[0].rank().corr(d[1].rank()) if len(d) > 2 else np.nan


def save_fig(fig, name):
    fig.savefig(OUT / "figures" / f"{name}.png", bbox_inches="tight")
    fig.savefig(OUT / "figures" / f"{name}.pdf", bbox_inches="tight")

## 11. Point sample

Besides the script's own layers, four **independent** layers are sampled. None of them enters the index, so they can be used to check it:

- `gain_2000_2012`: observed tree-cover gain (Hansen)
- `treecover2000`: tree canopy cover in 2000 (Hansen)
- `ndvi_trend`: Sen's slope of annual mean MODIS NDVI, 2001-2024 (greening / browning)
- `biome`, `protected`, `adm1_code`: RESOLVE biome, WDPA protected area flag, GAUL admin-1 unit

In [ ]:
def aux_layers(study_area):
    """Independent layers used for validation and stratification (not in the index)."""
    gfc = ee.Image("UMD/hansen/global_forest_change_2024_v1_12")
    gain = gfc.select("gain").rename("gain_2000_2012")
    treecover = gfc.select("treecover2000")

    modis = ee.ImageCollection("MODIS/061/MOD13Q1").select("NDVI")

    def annual(y):
        y = ee.Number(y)
        ndvi_y = modis.filter(ee.Filter.calendarRange(y, y, "year")).mean().multiply(0.0001)
        return ee.Image.constant(y).toFloat().rename("year").addBands(ndvi_y.rename("ndvi"))

    ndvi_trend = (
        ee.ImageCollection(ee.List.sequence(2001, 2024).map(annual))
        .reduce(ee.Reducer.sensSlope())
        .select("slope")
        .rename("ndvi_trend")
    )

    biome = (
        ee.FeatureCollection("RESOLVE/ECOREGIONS/2017")
        .filterBounds(study_area)
        .reduceToImage(["BIOME_NUM"], ee.Reducer.first())
        .rename("biome")
    )
    protected = (
        ee.Image.constant(0).byte()
        .paint(ee.FeatureCollection("WCMC/WDPA/current/polygons").filterBounds(study_area), 1)
        .rename("protected")
    )
    adm1 = (
        ee.FeatureCollection("FAO/GAUL/2015/level1")
        .filterBounds(study_area)
        .reduceToImage(["ADM1_CODE"], ee.Reducer.first())
        .rename("adm1_code")
    )
    return [gain, treecover, ndvi_trend, biome, protected, adm1]


def sample_stack(r):
    """One multi-band image with everything that is sampled."""
    bands = (
        list(r["factors"].values())
        + list(r["normalized"].values())
        + [r["index"], r["classified"], r["extras"]["elevation"], r["extras"]["soilPh"]]
        + aux_layers(r["study_area"])
        + [ee.Image.pixelLonLat()]
    )
    return ee.Image.cat(bands)


def sample_country(name, r, n=N_SAMPLES, chunk=SAMPLE_CHUNK, scale=SAMPLE_SCALE):
    """Random points -> DataFrame. Cached per chunk, so an interrupted run resumes."""
    stack = sample_stack(r)
    region = r["study_area"].geometry()
    tag = name.replace(" ", "_")

    def get_chunk(seed):
        path = OUT / "data" / f"sample_{tag}_s{scale}_c{chunk}_{seed:03d}.csv"
        if not path.exists():
            points = ee.FeatureCollection.randomPoints(region, chunk, seed)
            fc = stack.reduceRegions(points, ee.Reducer.first(), scale, tileScale=4)
            rows = [f["properties"] for f in fc.getInfo()["features"]]
            pd.DataFrame(rows).to_csv(path, index=False)
        return pd.read_csv(path)

    frames = run_parallel(get_chunk, range(int(np.ceil(n / chunk))), workers=4,
                          label=f"{name} sample chunk")
    frames = [f for f in frames if f is not None]
    df = pd.concat(frames, ignore_index=True)
    df.insert(0, "country", name)
    return df

In [ ]:
sample_frames = []
for name, r in results.items():
    t0 = time.time()
    df = sample_country(name, r)
    sample_frames.append(df)
    print(f"{name}: {len(df)} points ({time.time() - t0:.0f} s)")

samples = pd.concat(sample_frames, ignore_index=True)
samples.to_csv(OUT / "data" / "samples_all_countries.csv", index=False)

FACTORS = list(weights)                       # 20 factor names
NORM_COLS = [f + "_norm" for f in FACTORS]    # their normalised columns
INDEX = "ForestLandRestoration"
samples.groupby("country")[FACTORS + [INDEX]].count().T

## 12. Variable atlas: all variables for each country

One figure per country with the 20 input variables. The FLR index is not in the atlas: as the main result it has its own figure, with all countries together, right after. Continuous variables are stretched between the 2nd and 98th percentile of that country's sample, so every panel uses its full colour range (the fixed ranges of the Code Editor script left several layers almost blank). Figures are saved as PNG (600 dpi) and PDF in `outputs/figures/`.

Every map has the country border drawn on top. Display choices that differ from the raw data: **GRACE** cells are about 100 km wide, so the panel is interpolated for readability; **settlements** are drawn as built-up / not built-up with built-up pixels enlarged by two map pixels, because single 10 m settlement pixels vanish at country scale.

Downloaded map pictures are cached in `outputs/cache/`, so changing the layout or re-running is fast. Delete that folder to download them again.

In [ ]:
SEQ = {
    "green": ["#ffffe5", "#d9f0a3", "#78c679", "#238443", "#004529"],
    "blue": ["#f7fbff", "#c6dbef", "#6baed6", "#2171b5", "#08306b"],
    "red": ["#ffffcc", "#fed976", "#fd8d3c", "#e31a1c", "#800026"],
    "brown": ["#fff7bc", "#fec44f", "#d95f0e", "#993404", "#662506"],
    "purple": ["#fcfbfd", "#dadaeb", "#9e9ac8", "#6a51a3", "#3f007d"],
    "grey": ["#ffffff", "#d9d9d9", "#969696", "#525252", "#000000"],
    "div": ["#8c510a", "#d8b365", "#f5f5f5", "#5ab4ac", "#01665e"],
}
FLR_PALETTE = flr_vis["palette"]

# (column, panel title, unit, palette)  -> continuous, percentile stretch
CONTINUOUS = [
    ("ndvi", "NDVI", "-", SEQ["green"]),
    ("bsi", "Bare soil index", "-", SEQ["brown"]),
    ("ndwi", "NDWI (water index)", "-", SEQ["blue"]),
    ("nbai", "Built-up index (NBAI)", "-", SEQ["purple"]),
    ("precipitation", "Precipitation 2024", "mm", SEQ["blue"]),
    ("maxTemperature", "Max temperature", "deg C", SEQ["red"]),
    ("soilMoisture", "Soil moisture", "TerraClimate units", SEQ["blue"]),
    ("undergroundWater", "Groundwater (GRACE)", "cm", SEQ["blue"]),
    ("slope", "Slope", "deg", SEQ["red"]),
    ("drainage", "Upstream area", "km2", SEQ["blue"]),
    ("erosion", "Erosion (RUSLE)", "relative", SEQ["brown"]),
    ("soc", "Soil organic carbon", "script units", SEQ["brown"]),
    ("biomasC", "Biomass carbon", "t/ha", SEQ["green"]),
    ("population_density", "Population density", "persons/km2", SEQ["red"]),
    ("treeloss", "Tree loss (year, 0 = none)", "years since 2000", SEQ["red"]),
]

WORLDCOVER = {
    10: ("Tree cover", "#006400"), 20: ("Shrubland", "#ffbb22"), 30: ("Grassland", "#ffff4c"),
    40: ("Cropland", "#f096ff"), 50: ("Built-up", "#fa0000"), 60: ("Bare / sparse", "#b4b4b4"),
    70: ("Snow / ice", "#f0f0f0"), 80: ("Water", "#0064c8"), 90: ("Wetland", "#0096a0"),
    95: ("Mangroves", "#00cfa5"), 100: ("Moss / lichen", "#fae6a0"),
}

# (column, title, vis params) -> categorical, fixed range
CATEGORICAL = [
    ("soilTexture", "Soil texture class (1-12)",
     {"min": 1, "max": 12, "palette": ["#fef0d9", "#fdd49e", "#fdb863", "#fb8d3c", "#e6550d", "#a63603"]}),
    ("fireMask", "MODIS fire mask (7-9 = fire)",
     {"min": 0, "max": 9, "palette": ["#f0f0f0"] * 7 + ["#fdae61", "#f46d43", "#a50026"]}),
    ("cropMask", "Crop mask (GFSAD, 0-5)",
     {"min": 0, "max": 5, "palette": ["#f0f0f0", "#ffa500", "#a52a2a", "#02a50f", "#008000", "#ffff00"]}),
]


def _fetch_thumb_live(image, vis, extent, px, tile_deg=3.0, tile_cache=None):
    """RGBA picture of an image over extent [x0, x1, y0, y1], fetched as a mosaic of small requests."""
    x0, x1, y0, y1 = extent
    nx = int(np.ceil((x1 - x0) / tile_deg))
    ny = int(np.ceil((y1 - y0) / tile_deg))
    res = max(x1 - x0, y1 - y0) / px                      # degrees per pixel
    tw = max(8, int(round((x1 - x0) / nx / res)))
    th = max(8, int(round((y1 - y0) / ny / res)))
    vis_image = image.visualize(**vis)
    dx, dy = (x1 - x0) / nx, (y1 - y0) / ny

    def get(ij):
        i, j = ij
        # every downloaded tile is kept on disk, so a retry only fetches the tiles that failed
        f = tile_cache.with_name(f"{tile_cache.stem}_{i}_{j}.npy") if tile_cache else None
        if f is not None and f.exists():
            return np.load(f)
        rect = ee.Geometry.Rectangle(
            [x0 + i * dx, y1 - (j + 1) * dy, x0 + (i + 1) * dx, y1 - j * dy], None, False)
        url = vis_image.getThumbURL(
            {"region": rect, "dimensions": f"{tw}x{th}", "format": "png", "crs": "EPSG:4326"})
        resp = requests.get(url, timeout=600)
        resp.raise_for_status()
        tile = np.array(Image.open(io.BytesIO(resp.content)).convert("RGBA").resize((tw, th)))
        if f is not None:
            np.save(f, tile)
        return tile

    cells = [(i, j) for j in range(ny) for i in range(nx)]
    tiles = run_parallel(get, cells, workers=4, retries=5, label="map tile")
    if any(t is None for t in tiles):
        raise RuntimeError("some map tiles could not be downloaded")
    rows = [np.hstack([tiles[j * nx + i] for i in range(nx)]) for j in range(ny)]
    return np.vstack(rows)


def fetch_thumb(image, vis, extent, px=THUMB_PX, cache_key=None):
    """Map picture, cached on disk under outputs/cache when cache_key is given.

    The key should describe the layer (country + layer name); the colour settings are added
    to it automatically. Delete outputs/cache to force a fresh download.
    """
    path = None
    if cache_key:
        digest = hashlib.md5(json.dumps([cache_key, vis, px], sort_keys=True).encode()).hexdigest()[:16]
        path = OUT / "cache" / f"{digest}.npy"
        if path.exists():
            return np.load(path)
    img = _fetch_thumb_live(image, vis, extent, px, tile_cache=path)
    if path is not None:
        np.save(path, img)
    return img


SETTLEMENT_COLORS = {"Not built-up": "#ececec", "Built-up": "#d7191c"}


def atlas_panels(name):
    """Panel specs for the 20 input variables of one country."""
    r = results[name]
    d = samples[samples["country"] == name]
    panels = []
    for col, title, unit, pal in CONTINUOUS:
        image = r["factors"][col]
        if col == "undergroundWater":
            # GRACE cells are about 100 km wide: interpolate for display and use the full value range
            image = (ee.Image("NASA/GRACE/MASS_GRIDS_V04/LAND/20170501_20170522")
                     .select("lwe_thickness_csr").resample("bicubic"))
            lo, hi = d[col].min(), d[col].max()
        else:
            lo, hi = d[col].quantile([0.02, 0.98])
        if not np.isfinite(lo) or lo == hi:
            lo, hi = d[col].min(), d[col].max() + 1e-9
        panels.append(dict(key=col, image=image, vis={"min": float(lo), "max": float(hi), "palette": pal},
                           title=title, unit=unit, kind="continuous"))
    codes = list(WORLDCOVER)
    panels.append(dict(
        key="lulc", image=r["factors"]["lulc"].remap(codes, list(range(len(codes)))),
        vis={"min": 0, "max": len(codes) - 1, "palette": [c for _, c in WORLDCOVER.values()]},
        title="Land cover (WorldCover)", kind="legend"))
    for col, title, vis in CATEGORICAL:
        panels.append(dict(key=col, image=r["factors"][col], vis=vis, title=title, unit="class",
                           kind="continuous"))
    # Settlements: built-up classes (11-25), enlarged by 2 map pixels so towns show at country scale
    built = r["factors"]["settlement"].gte(11).focal_max(2, "circle", "pixels")
    panels.append(dict(key="settlement_builtup_r2", image=built,
                       vis={"min": 0, "max": 1, "palette": list(SETTLEMENT_COLORS.values())},
                       title="Settlements (GHSL built-up)", kind="legend"))
    return panels


def country_extent(name):
    xs, ys = zip(*results[name]["study_area"].geometry().bounds().coordinates().getInfo()[0])
    extent = [min(xs), max(xs), min(ys), max(ys)]
    aspect = 1 / np.cos(np.deg2rad((extent[2] + extent[3]) / 2))
    return extent, aspect


_OUTLINES = {}


def country_outline(name):
    """Border of a study area as a list of (lon, lat) rings, simplified to 500 m and kept in memory."""
    if name not in _OUTLINES:
        geom = results[name]["study_area"].geometry().simplify(500).getInfo()

        def rings(g):
            if g["type"] == "Polygon":
                return list(g["coordinates"])
            if g["type"] == "MultiPolygon":
                return [ring for poly in g["coordinates"] for ring in poly]
            if g["type"] == "GeometryCollection":
                return [ring for part in g["geometries"] for ring in rings(part)]
            return []

        _OUTLINES[name] = [np.array(ring) for ring in rings(geom)]
    return _OUTLINES[name]


def draw_outline(ax, name, color="black", linewidth=0.5):
    """Draw the country border on a map panel."""
    for ring in country_outline(name):
        ax.plot(ring[:, 0], ring[:, 1], color=color, linewidth=linewidth, solid_joinstyle="round",
                scalex=False, scaley=False, zorder=5)


def plot_atlas(name, ncols=5):
    r = results[name]
    extent, aspect = country_extent(name)
    panels = atlas_panels(name)

    thumbs = []
    for p in panels:
        try:
            thumbs.append(fetch_thumb(p["image"].clip(r["study_area"]), p["vis"], extent,
                                      cache_key=f"{name}/{p['key']}" + ("/v2" if p["key"] == "treeloss" else "")))
        except Exception as e:
            print(f"  {name} / {p['title']}: {str(e)[:120]}")
            thumbs.append(None)

    nrows = int(np.ceil(len(panels) / ncols))
    ratio = (extent[3] - extent[2]) * aspect / (extent[1] - extent[0])
    fig, axes = plt.subplots(nrows, ncols, figsize=(7.2, 7.2 / ncols * nrows * (ratio + 0.38) + 0.5))
    for ax, p, img in zip(axes.flat, panels, thumbs):
        if img is None:
            ax.text(0.5, 0.5, "not available", ha="center", va="center", transform=ax.transAxes)
        else:
            ax.imshow(img, extent=extent, aspect=aspect, interpolation="nearest")
            draw_outline(ax, name)
        ax.set_title(p["title"], pad=2)
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values():
            s.set_visible(False)
        if p["kind"] == "continuous":
            cmap = LinearSegmentedColormap.from_list("c", p["vis"]["palette"])
            sm = ScalarMappable(cmap=cmap)
            sm.set_clim(p["vis"]["min"], p["vis"]["max"])
            cb = fig.colorbar(sm, ax=ax, orientation="horizontal", fraction=0.05, pad=0.03)
            cb.ax.tick_params(length=1.5, pad=1)
            cb.ax.locator_params(nbins=3)
            cb.set_label(p["unit"], labelpad=1)
            cb.outline.set_linewidth(0.3)
    for ax in list(axes.flat)[len(panels):]:
        ax.axis("off")
    fig.suptitle(name, fontsize=9, fontweight="bold", y=0.995)
    legend_h = 0.6 / fig.get_figheight()
    fig.tight_layout(pad=0.3, rect=[0, legend_h, 1, 0.975])
    fig.legend(handles=[Patch(color=c, label=l) for l, c in WORLDCOVER.values()],
               title="Land cover", loc="lower left", bbox_to_anchor=(0.01, 0), ncol=6,
               frameon=False, fontsize=5, title_fontsize=6, columnspacing=1, handlelength=1.2)
    fig.legend(handles=[Patch(color=c, label=l) for l, c in SETTLEMENT_COLORS.items()],
               title="Settlements", loc="lower right", bbox_to_anchor=(0.99, 0), ncol=1,
               frameon=False, fontsize=5, title_fontsize=6, handlelength=1.2)
    save_fig(fig, f"atlas_{name.replace(' ', '_')}")
    return fig

In [ ]:
for name in results:
    print("atlas:", name)
    plot_atlas(name)
    plt.show()

### FLR index: all countries together

The main result. Each country has an equal-sized panel and all five use **one shared colour scale** (2nd-98th percentile of the pooled sample), so colours can be compared directly. The map scale differs between panels because the countries differ greatly in size; each panel has its own scale bar. Keep in mind that this is the index with per-country normalisation; the cross-country comparable versions are `index_pooled` (section 19) and `FLR_need` (section 22).

In [ ]:
def draw_scale_bar(ax, extent):
    """Scale bar in the lower-left corner, with a round length of about a quarter of the map width."""
    x0, x1, y0, y1 = extent
    km_per_deg = 111.32 * np.cos(np.deg2rad((y0 + y1) / 2))
    target = (x1 - x0) * km_per_deg / 4
    length = max(k for k in [25, 50, 100, 200, 300, 500, 1000] if k <= max(target, 25))
    bx, by = x0 + 0.03 * (x1 - x0), y0 + 0.02 * (y1 - y0)
    ax.plot([bx, bx + length / km_per_deg], [by, by], color="black", linewidth=1.2,
            solid_capstyle="butt", scalex=False, scaley=False, zorder=6)
    ax.text(bx, by + 0.015 * (y1 - y0), f"{length} km", fontsize=5, va="bottom", ha="left", zorder=6)


def plot_all_countries(layer_key, vis, label, fname, px=900, cache_tag=None, classes=None, ncols=3):
    """One equal-sized panel per country, each country filling its panel, one shared colour scale.

    Because the countries differ a lot in size, each panel has its own map scale (see the scale
    bars); the colours are directly comparable. layer_key: key in results[country] ("index",
    "need", ...). classes: {label: colour} for a categorical layer, otherwise a colour bar.
    """
    names = list(results)
    nrows = int(np.ceil((len(names) + 1) / ncols))      # one spare cell for the legend
    side = 7.2 / ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(7.2, side * nrows + 0.2))
    axes = np.atleast_2d(axes)
    for ax, name in zip(axes.flat, names):
        r = results[name]
        extent, aspect = country_extent(name)
        try:
            img = fetch_thumb(r[layer_key].clip(r["study_area"]), vis, extent, px=px,
                              cache_key=f"{name}/{cache_tag or layer_key}/v2")
            ax.imshow(img, extent=extent, aspect=aspect, interpolation="nearest")
            draw_outline(ax, name)
        except Exception as e:
            print(f"  {name}: {str(e)[:120]}")
            ax.set_xlim(extent[0], extent[1]); ax.set_ylim(extent[2], extent[3]); ax.set_aspect(aspect)
        draw_scale_bar(ax, extent)
        ax.set_title(name, pad=3, fontsize=8, fontweight="bold")
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values():
            s.set_visible(False)
        print("mapped:", name)

    spare = list(axes.flat)[len(names):]
    for ax in spare:
        ax.axis("off")
    fig.tight_layout(pad=0.4)
    key_ax = spare[0]
    if classes:
        key_ax.legend(handles=[Patch(color=col, label=lab) for lab, col in classes.items()],
                      title=label, loc="center", frameon=False, fontsize=7, title_fontsize=7)
    else:
        sm = ScalarMappable(cmap=LinearSegmentedColormap.from_list("c", vis["palette"]))
        sm.set_clim(vis["min"], vis["max"])
        box = key_ax.get_position()
        cax = fig.add_axes([box.x0 + 0.1 * box.width, box.y0 + 0.5 * box.height,
                            0.8 * box.width, 0.06 * box.height])
        cb = fig.colorbar(sm, cax=cax, orientation="horizontal")
        cb.set_label(label); cb.outline.set_linewidth(0.3)
        key_ax.text(0.5, 0.36, "Same colour scale in all panels;\nmap scale differs (see scale bars).",
                    ha="center", va="top", fontsize=6, color="0.3", transform=key_ax.transAxes)
    save_fig(fig, fname)
    return fig


flr_lo, flr_hi = samples[INDEX].quantile([0.02, 0.98])
flr_shared_vis = {"min": float(flr_lo), "max": float(flr_hi), "palette": FLR_PALETTE}
plot_all_countries("index", flr_shared_vis, "ForestLandRestoration index",
                   "flr_index_all_countries")
plt.show()

## 13. Descriptive statistics

Median and inter-quartile range of every variable in every country.

In [ ]:
def med_iqr(s):
    q = s.quantile([0.25, 0.5, 0.75])
    return f"{q[0.5]:.3g} ({q[0.25]:.3g}-{q[0.75]:.3g})"

CONT_COLS = [c for c, *_ in CONTINUOUS] + ["elevation", "soilPh", INDEX]
table1 = samples.groupby("country")[CONT_COLS].agg(med_iqr).T
table1.to_csv(OUT / "data" / "table1_descriptive.csv")
table1

### Distribution of the index per country

In [ ]:
index_summary = (samples.groupby("country")[INDEX]
                 .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).loc[list(results)])
index_summary.to_csv(OUT / "data" / "flr_index_summary.csv")

fig, ax = plt.subplots(figsize=(4.5, 2.2))
y = range(len(index_summary))
ax.hlines(y, index_summary["10%"], index_summary["90%"], color="#9ecae1", linewidth=2, label="p10 - p90")
ax.hlines(y, index_summary["25%"], index_summary["75%"], color="#3182bd", linewidth=5, label="p25 - p75")
ax.plot(index_summary["50%"], y, "o", ms=3, color="white", markeredgecolor="#08306b", label="median")
ax.set_yticks(list(y)); ax.set_yticklabels(index_summary.index)
ax.invert_yaxis()
ax.set_xlabel("ForestLandRestoration index (per-country normalisation)")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="lower right")
save_fig(fig, "index_distribution")
plt.show()
index_summary.round(3)

## 14. Collinearity between factors

A weighted sum counts correlated factors more than once. Spearman correlations and variance inflation factors (VIF) on the normalised factors show where that happens. As a rule of thumb, |rho| > 0.7 or VIF > 5 means two factors carry largely the same information, and one of them should be dropped or the weights merged.

In [ ]:
complete = samples.dropna(subset=NORM_COLS + [INDEX]).reset_index(drop=True)
print("points with all factors:", len(complete), "of", len(samples))

Xn = complete[NORM_COLS].rename(columns=lambda c: c.replace("_norm", ""))
Xn = Xn.loc[:, Xn.std() > 0]
rho = Xn.rank().corr()   # Spearman

fig, ax = plt.subplots(figsize=(5.2, 4.6))
im = ax.imshow(rho, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(rho))); ax.set_xticklabels(rho.columns, rotation=90)
ax.set_yticks(range(len(rho))); ax.set_yticklabels(rho.index)
for i in range(len(rho)):
    for j in range(len(rho)):
        if i != j and abs(rho.iat[i, j]) >= 0.5:
            ax.text(j, i, f"{rho.iat[i, j]:.1f}", ha="center", va="center", fontsize=4.5,
                    color="white" if abs(rho.iat[i, j]) > 0.75 else "black")
fig.colorbar(im, ax=ax, fraction=0.046, label="Spearman rho")
ax.set_title("Correlation between normalised factors (all countries pooled)")
save_fig(fig, "factor_correlation")
plt.show()

# VIF = diagonal of the inverse Pearson correlation matrix
vif = pd.Series(np.diag(np.linalg.pinv(Xn.corr().to_numpy())), index=Xn.columns, name="VIF")
pairs = rho.where(np.triu(np.ones(rho.shape, bool), 1)).stack()
print("Pairs with |rho| > 0.7:")
print(pairs[pairs.abs() > 0.7].round(2).to_string() or "  none")
vif.sort_values(ascending=False).round(2).to_frame()

## 15. PCA-derived weights compared with AHP

Principal components with eigenvalue > 1 are retained. The data-driven weight of a factor is the sum of its squared loadings weighted by the variance each component explains (the OECD composite-indicator method). Large gaps between AHP and PCA weights point at factors whose expert weight is not supported by the structure of the data, and need to be argued for explicitly.

In [ ]:
Z = (Xn - Xn.mean()) / Xn.std()
eigval, eigvec = np.linalg.eigh(np.corrcoef(Z.to_numpy(), rowvar=False))
order = np.argsort(eigval)[::-1]
eigval, eigvec = eigval[order], eigvec[:, order]
keep = eigval > 1
print(f"components kept: {keep.sum()}, variance explained: {eigval[keep].sum() / eigval.sum():.1%}")

pca_w = (eigvec[:, keep] ** 2 * (eigval[keep] / eigval[keep].sum())).sum(axis=1)
w_ahp = pd.Series(weights)[Xn.columns]
compare_w = pd.DataFrame({
    "AHP": w_ahp / w_ahp.sum(),
    "PCA": pd.Series(pca_w / pca_w.sum(), index=Xn.columns),
}).sort_values("AHP")

fig, ax = plt.subplots(figsize=(3.6, 4.2))
y = np.arange(len(compare_w))
ax.barh(y - 0.2, compare_w["AHP"], 0.4, label="AHP (expert)", color="#2171b5")
ax.barh(y + 0.2, compare_w["PCA"], 0.4, label="PCA (data-driven)", color="#fd8d3c")
ax.set_yticks(y); ax.set_yticklabels(compare_w.index)
ax.set_xlabel("Weight (rescaled to sum to 1)")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
save_fig(fig, "weights_ahp_vs_pca")
plt.show()

# Rank agreement of the two resulting indices
idx_ahp = Xn.to_numpy() @ compare_w["AHP"].reindex(Xn.columns).to_numpy()
idx_pca = Xn.to_numpy() @ compare_w["PCA"].reindex(Xn.columns).to_numpy()
print("Spearman rho between AHP index and PCA index:",
      round(spearman(idx_ahp, idx_pca), 3))

## 16. Monte Carlo uncertainty of the weights

AHP weights are judgements, not measurements. Each weight is perturbed with log-normal noise (`MC_SIGMA`), the vector is rescaled to the original sum, and the index and classes are recomputed `N_MC` times. Outputs:

- **class stability**: share of draws in which a point keeps its baseline class
- **class shares with 95 % interval** per country

The first line checks that the index recomputed locally equals the one from Earth Engine.

In [ ]:
X = complete[NORM_COLS].to_numpy(dtype="float32")
w0 = np.array([weights[f] for f in FACTORS], dtype="float32")


def to_class(v):
    return np.digitize(v, [0.25, 0.50, 0.75], right=True) + 1


base = X @ w0
print("max |local index - Earth Engine index|:", float(np.abs(base - complete[INDEX]).max()))
base_cls = to_class(base)

rng = np.random.default_rng(42)
W = w0 * rng.lognormal(0, MC_SIGMA, size=(N_MC, len(w0))).astype("float32")
W *= w0.sum() / W.sum(axis=1, keepdims=True)
mc = X @ W.T                      # points x draws
mc_cls = to_class(mc)

complete["index_local"] = base
complete["index_mc_sd"] = mc.std(axis=1)
complete["class_stability"] = (mc_cls == base_cls[:, None]).mean(axis=1)

rows = []
for name in results:
    m = (complete["country"] == name).to_numpy()
    for c, label in CLASS_NAMES.items():
        share = (mc_cls[m] == c).mean(axis=0) * 100
        rows.append({"country": name, "class": label, "baseline_%": (base_cls[m] == c).mean() * 100,
                     "mc_median_%": np.median(share), "mc_2.5_%": np.percentile(share, 2.5),
                     "mc_97.5_%": np.percentile(share, 97.5)})
mc_shares = pd.DataFrame(rows)
mc_shares.to_csv(OUT / "data" / "mc_class_shares.csv", index=False)

print("Mean class stability per country:")
print(complete.groupby("country")["class_stability"].mean().round(3).to_string())
mc_shares.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 2.6))
names = list(results)
width = 0.2
for k, (c, label) in enumerate(CLASS_NAMES.items()):
    d = mc_shares[mc_shares["class"] == label].set_index("country").loc[names]
    x = np.arange(len(names)) + (k - 1.5) * width
    err = [d["mc_median_%"] - d["mc_2.5_%"], d["mc_97.5_%"] - d["mc_median_%"]]
    ax.bar(x, d["mc_median_%"], width, yerr=err, color=class_colors[k], label=label,
           error_kw={"elinewidth": 0.6, "capsize": 1.5})
ax.set_xticks(np.arange(len(names))); ax.set_xticklabels(names)
ax.set_ylabel("Share of area (%)")
ax.legend(title="Class", frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.25))
ax.spines[["top", "right"]].set_visible(False)
save_fig(fig, "mc_class_shares")
plt.show()

## 17. One-at-a-time sensitivity

Each weight is changed by +/-20 % (others rescaled so the sum is unchanged). The table reports how many points change class. Factors at the top are the ones whose weight must be justified most carefully.

In [ ]:
rows = []
for k, f in enumerate(FACTORS):
    for delta in (-0.2, 0.2):
        w = w0.copy()
        w[k] *= 1 + delta
        others = np.arange(len(w)) != k
        w[others] *= (w0.sum() - w[k]) / w0[others].sum()
        idx = X @ w
        rows.append({"factor": f, "change": f"{delta:+.0%}",
                     "points_changing_class_%": (to_class(idx) != base_cls).mean() * 100,
                     "mean_index_shift": float((idx - base).mean())})
oat = pd.DataFrame(rows)
oat_summary = (oat.groupby("factor")["points_changing_class_%"].max()
               .sort_values(ascending=False).rename("max_points_changing_class_%"))
oat.to_csv(OUT / "data" / "oat_sensitivity.csv", index=False)
oat_summary.round(2).to_frame()

## 18. What drives the index in each country

Mean contribution of each factor (`weight x normalised value`) as a share of the mean index.

In [ ]:
contrib = pd.DataFrame(X * w0, columns=FACTORS)
contrib["country"] = complete["country"].to_numpy()
contrib_share = contrib.groupby("country").mean()
contrib_share = contrib_share.div(contrib_share.sum(axis=1), axis=0) * 100
contrib_share = contrib_share.loc[list(results), contrib_share.mean().sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(6.5, 2.2))
im = ax.imshow(contrib_share, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(contrib_share.shape[1])); ax.set_xticklabels(contrib_share.columns, rotation=90)
ax.set_yticks(range(contrib_share.shape[0])); ax.set_yticklabels(contrib_share.index)
for i in range(contrib_share.shape[0]):
    for j in range(contrib_share.shape[1]):
        v = contrib_share.iat[i, j]
        if v >= 3:
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=5,
                    color="white" if v > contrib_share.to_numpy().max() * 0.6 else "black")
fig.colorbar(im, ax=ax, fraction=0.025, label="Share of mean index (%)")
save_fig(fig, "factor_contributions")
plt.show()
contrib_share.round(1)

## 19. Pooled normalisation: a scale that is comparable across countries

In the original design, min and max are taken inside each study area, so "0.6 in Togo" and "0.6 in Ethiopia" do not mean the same thing. Here every factor is rescaled with **one** pair of bounds for all five countries (2nd and 98th percentile of the pooled sample, clipped to 0-1, which also removes the influence of single outlier pixels). The resulting `index_pooled` can be compared between countries.

In [ ]:
raw = complete[FACTORS].copy()
raw["lulc"] = raw["lulc"] / 100          # same rule as the script
lo, hi = raw.quantile(0.02), raw.quantile(0.98)
lo["lulc"], hi["lulc"] = 0, 1
pooled_norm = ((raw - lo) / (hi - lo).replace(0, 1)).clip(0, 1)
complete["index_pooled"] = pooled_norm.to_numpy() @ w0

bounds = pd.DataFrame({"p02": lo, "p98": hi})
bounds.to_csv(OUT / "data" / "pooled_normalisation_bounds.csv")

fig, axes = plt.subplots(1, 2, figsize=(6.5, 2.4), sharey=True)
for ax, col, title in zip(axes, ["index_local", "index_pooled"],
                          ["Per-country normalisation (original)", "Pooled normalisation"]):
    data = [complete.loc[complete["country"] == n, col] for n in results]
    bp = ax.boxplot(data, vert=False, showfliers=False, patch_artist=True, widths=0.6,
                    medianprops={"color": "black"})
    for b in bp["boxes"]:
        b.set(facecolor="#9ecae1", linewidth=0.5)
    ax.set_yticks(range(1, len(results) + 1)); ax.set_yticklabels(list(results))
    ax.set_title(title); ax.set_xlabel("FLR index")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].invert_yaxis()
save_fig(fig, "index_per_country_vs_pooled")
plt.show()

print("Spearman rho between the two indices, within each country:")
print(complete.groupby("country").apply(
    lambda d: spearman(d["index_local"], d["index_pooled"])).round(3).to_string())

## 20. Validation against independent observations

Two checks with data that do not enter the index:

1. **Observed tree-cover gain (Hansen, 2000-2012).** AUC of the index as a predictor of where gain happened. AUC = 0.5 means no relationship; above 0.5 means gain occurred where the index is high, below 0.5 where it is low.
2. **MODIS NDVI trend 2001-2024.** Spearman correlation between the index and the greening/browning slope.

95 % intervals come from 500 bootstrap resamples. This is an exploratory check, not a full validation: tree-cover gain mixes plantations with natural regrowth, and the direction you *expect* depends on whether a high index is meant to flag constraint or opportunity. State that expectation before looking at the numbers.

In [ ]:
def auc(score, label):
    score, label = np.asarray(score, float), np.asarray(label, bool)
    n1, n0 = label.sum(), (~label).sum()
    if n1 == 0 or n0 == 0:
        return np.nan
    ranks = pd.Series(score).rank().to_numpy()
    return (ranks[label].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def bootstrap(fn, d, n=500, seed=0):
    rng = np.random.default_rng(seed)
    vals = [fn(d.iloc[rng.integers(0, len(d), len(d))]) for _ in range(n)]
    return np.nanpercentile(vals, [2.5, 97.5])


rows = []
for name in list(results) + ["All countries"]:
    d = complete if name == "All countries" else complete[complete["country"] == name]
    for col in ["index_local", "index_pooled"]:
        dg = d.dropna(subset=["gain_2000_2012"])
        f_auc = lambda x: auc(x[col], x["gain_2000_2012"] == 1)
        dt = d.dropna(subset=["ndvi_trend"])
        f_rho = lambda x: spearman(x[col], x["ndvi_trend"])
        a_lo, a_hi = bootstrap(f_auc, dg) if len(dg) else (np.nan, np.nan)
        r_lo, r_hi = bootstrap(f_rho, dt) if len(dt) else (np.nan, np.nan)
        rows.append({"country": name, "index": col,
                     "gain_points": int((dg["gain_2000_2012"] == 1).sum()),
                     "AUC_gain": f_auc(dg) if len(dg) else np.nan, "AUC_lo": a_lo, "AUC_hi": a_hi,
                     "rho_ndvi_trend": f_rho(dt) if len(dt) else np.nan, "rho_lo": r_lo, "rho_hi": r_hi})
validation = pd.DataFrame(rows)
validation.to_csv(OUT / "data" / "validation.csv", index=False)
validation.round(3)

## 21. Stratification: biome, protection status and admin-1 ranking

In [ ]:
BIOMES = {
    1: "Tropical moist broadleaf forest", 2: "Tropical dry broadleaf forest",
    3: "Tropical coniferous forest", 7: "Tropical grassland, savanna, shrubland",
    9: "Flooded grassland and savanna", 10: "Montane grassland and shrubland",
    12: "Mediterranean forest and scrub", 13: "Desert and xeric shrubland", 14: "Mangroves",
}
complete["biome_name"] = complete["biome"].map(BIOMES).fillna("Other / unclassified")
complete["class_name"] = pd.Series(base_cls, index=complete.index).map(CLASS_NAMES)


def strata_table(by):
    g = complete.groupby(by)
    out = g.agg(points=("index_local", "size"), index_mean=("index_local", "mean"),
                index_pooled_mean=("index_pooled", "mean"),
                class_stability=("class_stability", "mean"),
                ndvi_trend_median=("ndvi_trend", "median"))
    shares = (g["class_name"].value_counts(normalize=True).unstack(fill_value=0) * 100)
    return out.join(shares.reindex(columns=list(CLASS_NAMES.values()), fill_value=0).add_suffix("_%"))


by_biome = strata_table(["country", "biome_name"])
by_biome = by_biome[by_biome["points"] >= 30]
by_biome.to_csv(OUT / "data" / "strata_biome.csv")
by_biome.round(3)

In [ ]:
complete["protected_status"] = complete["protected"].map({1: "Protected", 0: "Not protected"})
by_protection = strata_table(["country", "protected_status"])
by_protection.to_csv(OUT / "data" / "strata_protection.csv")
by_protection.round(3)

In [ ]:
# Admin-1 names
adm1_names = {}
for name in results:
    fc = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(ee.Filter.eq("ADM0_NAME", name))
    info = ee.Dictionary.fromLists(
        fc.aggregate_array("ADM1_CODE").map(lambda c: ee.Number(c).format("%d")),
        fc.aggregate_array("ADM1_NAME")).getInfo()
    adm1_names.update({int(k): v for k, v in info.items()})

complete["adm1_name"] = complete["adm1_code"].map(adm1_names)
by_adm1 = strata_table(["country", "adm1_name"])
by_adm1 = by_adm1[by_adm1["points"] >= 30].sort_values(
    ["country", "index_pooled_mean"], ascending=[True, False])
by_adm1.to_csv(OUT / "data" / "strata_adm1.csv")
complete.to_csv(OUT / "data" / "samples_analysed.csv", index=False)
by_adm1.round(3)

## 22. Direction-aware index: degradation / restoration need

A second index, `FLR_need`, in which **a high value always means more degradation and a greater need for restoration**. It differs from the original index in four ways:

1. **Direction.** Each factor is given a sign. Factors where a high value means *less* degradation (NDVI, biomass, soil carbon, moisture, rainfall) are inverted.
2. **Categorical layers are scored, not scaled.** Land cover gets a need score per class; fire, cropland and settlement become present / absent.
3. **One normalisation for all countries** (the pooled 2nd-98th percentile bounds of section 19), so values are comparable between countries.
4. **Data-driven classes.** Four classes split at the quartiles of the pooled sample instead of fixed 0.25 steps.

Factors with direction `0` are left out and the remaining AHP weights are rescaled to sum to 1.

**The table below is a starting point that needs your judgement.** The signs and the land-cover scores are defaults chosen for this notebook, not values from the literature; review each line, change what you disagree with, and cite a source for every choice in the paper. Two factors are left out by default because their direction is not obvious: soil texture (needs a per-class score from a soil scientist) and upstream drainage area (large values are river channels, not degraded land).

In [ ]:
# +1: high value = more degradation / need     -1: high value = less     0: left out
DIRECTION = {
    "ndvi": -1,                # sparse vegetation = degraded
    "bsi": +1,                 # bare soil
    "ndwi": -1,                # dry surface
    "nbai": +1,                # built-up / bare signal
    "precipitation": -1,       # drier = more stressed
    "maxTemperature": +1,      # hotter = more stressed
    "soilMoisture": -1,
    "undergroundWater": -1,    # low water storage
    "slope": +1,               # steeper = more erosion-prone
    "erosion": +1,
    "soc": -1,                 # low soil organic carbon = degraded
    "biomasC": -1,             # low biomass carbon = degraded
    "population_density": +1,  # human pressure
    "treeloss": +1,            # 0 = no loss, larger = more recent loss
    "fireMask": +1,            # scored as fire / no fire
    "cropMask": +1,            # scored as cropland / not cropland
    "settlement": +1,          # scored as built / not built
    "lulc": +1,                # scored per class, see LULC_NEED
    "soilTexture": 0,          # no defensible direction without per-class scores
    "drainage": 0,             # upstream area: not a degradation measure
}

# Need score per WorldCover class (0 = no need / not restorable, 1 = highest need)
LULC_NEED = {
    10: 0.1,   # tree cover
    20: 0.4,   # shrubland
    30: 0.5,   # grassland
    40: 0.7,   # cropland
    50: 0.9,   # built-up
    60: 1.0,   # bare / sparse vegetation
    70: 0.0,   # snow / ice
    80: 0.0,   # water
    90: 0.1,   # herbaceous wetland
    95: 0.1,   # mangroves
    100: 0.3,  # moss / lichen
}

# Categorical layers turned into present (1) / absent (0): value >= threshold
BINARY_MIN = {
    "fireMask": 7,     # MODIS fire mask 7-9 = fire
    "cropMask": 1,     # GFSAD 1-5 = cropland
    "settlement": 11,  # GHSL built characteristics 11-25 = built-up
}

kept = [f for f in FACTORS if DIRECTION[f] != 0]
w_need = pd.Series({f: weights[f] for f in kept})
w_need = w_need / w_need.sum()

pd.DataFrame({
    "direction": pd.Series(DIRECTION).map({1: "+ (more need)", -1: "- (inverted)", 0: "left out"}),
    "scoring": pd.Series({f: "class score" if f == "lulc" else "present / absent" if f in BINARY_MIN
                          else "pooled p2-p98" for f in FACTORS}),
    "AHP weight": pd.Series(weights),
    "weight used": w_need.reindex(FACTORS).fillna(0).round(3),
})

In [ ]:
def need_scores(df):
    """0-1 need score of every kept factor for a table of raw values."""
    s = pd.DataFrame(index=df.index)
    for f in kept:
        if f == "lulc":
            s[f] = df[f].map(LULC_NEED)
        elif f in BINARY_MIN:
            s[f] = (df[f] >= BINARY_MIN[f]).astype(float)
        else:
            x = ((df[f] - lo[f]) / ((hi[f] - lo[f]) or 1)).clip(0, 1)
            s[f] = x if DIRECTION[f] > 0 else 1 - x
    return s


scores = need_scores(complete)
complete["index_need"] = scores.to_numpy() @ w_need.to_numpy()

NEED_BREAKS = complete["index_need"].quantile([0.25, 0.50, 0.75]).tolist()
NEED_CLASSES = {1: "Low", 2: "Moderate", 3: "High", 4: "Very high"}
NEED_COLORS = ["#ffffb2", "#fecc5c", "#fd8d3c", "#e31a1c"]
complete["need_class"] = np.digitize(complete["index_need"], NEED_BREAKS, right=True) + 1
print("class breaks (pooled quartiles):", [round(b, 3) for b in NEED_BREAKS])

need_summary = (complete.groupby("country")["index_need"]
                .describe(percentiles=[0.1, 0.5, 0.9]).loc[list(results)])
need_shares = (complete.groupby("country")["need_class"].value_counts(normalize=True)
               .unstack(fill_value=0).reindex(columns=[1, 2, 3, 4], fill_value=0)
               .rename(columns=NEED_CLASSES).loc[list(results)] * 100)
need_summary.to_csv(OUT / "data" / "need_index_summary.csv")
need_shares.to_csv(OUT / "data" / "need_class_shares_pct.csv")

print("\nShare of each country in each need class (%):")
display(need_shares.round(1))
need_summary.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.4))

need_shares.plot(kind="barh", stacked=True, color=NEED_COLORS, ax=axes[0], width=0.7,
                 edgecolor="white", linewidth=0.3)
axes[0].set_xlim(0, 100); axes[0].invert_yaxis()
axes[0].set_xlabel("Share of country area (%)"); axes[0].set_ylabel("")
axes[0].set_title("Restoration need classes")
axes[0].legend(frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.22))

data = [complete.loc[complete["country"] == n, "index_need"] for n in results]
bp = axes[1].boxplot(data, vert=False, showfliers=False, patch_artist=True, widths=0.6,
                     medianprops={"color": "black"})
for b in bp["boxes"]:
    b.set(facecolor="#fdae6b", linewidth=0.5)
for q in NEED_BREAKS:
    axes[1].axvline(q, color="grey", linewidth=0.4, linestyle="--")
axes[1].set_yticks(range(1, len(results) + 1)); axes[1].set_yticklabels(list(results))
axes[1].invert_yaxis()
axes[1].set_xlabel("FLR need index (dashed: class breaks)")
axes[1].set_title("Distribution per country")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
save_fig(fig, "need_index_by_country")
plt.show()

### What drives the need index, and how it relates to the original index and to observed greening

- **Contributions**: share of the mean need index coming from each factor, per country.
- **Agreement with the original index**: Spearman rank correlation. A low or negative value is expected, because the original index added "good" and "bad" factors with the same sign.
- **NDVI trend 2001-2024**: if the index captures degradation, land with a high need should be browning more than greening, so the correlation should be negative. NDVI *level* is an input but its *trend* is not, so this is an independent check.

In [ ]:
need_contrib = (scores * w_need).assign(country=complete["country"].to_numpy()).groupby("country").mean()
need_contrib = need_contrib.div(need_contrib.sum(axis=1), axis=0) * 100
need_contrib = need_contrib.loc[list(results), need_contrib.mean().sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(6.5, 2.2))
im = ax.imshow(need_contrib, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(need_contrib.shape[1])); ax.set_xticklabels(need_contrib.columns, rotation=90)
ax.set_yticks(range(need_contrib.shape[0])); ax.set_yticklabels(need_contrib.index)
for i in range(need_contrib.shape[0]):
    for j in range(need_contrib.shape[1]):
        v = need_contrib.iat[i, j]
        if v >= 3:
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=5,
                    color="white" if v > need_contrib.to_numpy().max() * 0.6 else "black")
fig.colorbar(im, ax=ax, fraction=0.025, label="Share of mean need index (%)")
save_fig(fig, "need_factor_contributions")
plt.show()

rows = []
for name in list(results) + ["All countries"]:
    d = complete if name == "All countries" else complete[complete["country"] == name]
    dt = d.dropna(subset=["ndvi_trend"])
    f_rho = lambda x: spearman(x["index_need"], x["ndvi_trend"])
    r_lo, r_hi = bootstrap(f_rho, dt)
    rows.append({"country": name,
                 "rho_with_original_index": spearman(d["index_need"], d["index_local"]),
                 "rho_ndvi_trend": f_rho(dt), "rho_lo": r_lo, "rho_hi": r_hi,
                 "browning_share_low_need_%": (dt.loc[dt["need_class"] == 1, "ndvi_trend"] < 0).mean() * 100,
                 "browning_share_very_high_need_%": (dt.loc[dt["need_class"] == 4, "ndvi_trend"] < 0).mean() * 100})
need_validation = pd.DataFrame(rows).set_index("country")
need_validation.to_csv(OUT / "data" / "need_validation.csv")
need_validation.round(3)

### Priority regions

Admin-1 units ranked by mean need index (units with at least 30 sample points), top five per country.

In [ ]:
g = complete.groupby(["country", "adm1_name"])
need_adm1 = g.agg(points=("index_need", "size"), need_mean=("index_need", "mean"),
                  very_high_share=("need_class", lambda s: (s == 4).mean() * 100),
                  ndvi_trend_median=("ndvi_trend", "median"))
need_adm1 = need_adm1[need_adm1["points"] >= 30].sort_values(
    ["country", "need_mean"], ascending=[True, False])
need_adm1.to_csv(OUT / "data" / "need_adm1_ranking.csv")
complete.to_csv(OUT / "data" / "samples_analysed.csv", index=False)
need_adm1.groupby(level="country").head(5).round(3)

### Maps of the need index

The same index as an Earth Engine image for each country (pooled bounds and class breaks from the sample), stored in `results[country]["need"]` and `results[country]["need_class"]`.

In [ ]:
def need_image(r):
    """FLR_need as an Earth Engine image, with the same scoring as need_scores()."""
    total = None
    for f, w in w_need.items():
        img = r["factors"][f]
        if f == "lulc":
            s = img.remap(list(LULC_NEED), list(LULC_NEED.values()))
        elif f in BINARY_MIN:
            s = img.gte(BINARY_MIN[f])
        else:
            rng = float(hi[f] - lo[f]) or 1.0
            s = img.subtract(float(lo[f])).divide(rng).clamp(0, 1)
            if DIRECTION[f] < 0:
                s = ee.Image.constant(1).subtract(s)
        term = s.multiply(float(w))
        total = term if total is None else total.add(term)
    return total.rename("FLR_need")


for name, r in results.items():
    need = need_image(r)
    r["need"] = need
    r["need_class"] = (need.gt(NEED_BREAKS[0]).add(need.gt(NEED_BREAKS[1]))
                       .add(need.gt(NEED_BREAKS[2])).add(1).toUint8().rename("FLR_need_class"))

need_lo, need_hi = complete["index_need"].quantile([0.02, 0.98])
need_vis = {"min": float(need_lo), "max": float(need_hi), "palette": SEQ["red"]}
need_class_vis = {"min": 1, "max": 4, "palette": NEED_COLORS}

In [ ]:
plot_all_countries("need", need_vis, "FLR need index (degradation / restoration need)",
                   "need_index_all_countries")
plt.show()
plot_all_countries("need_class", need_class_vis, "Need class", "need_classes_all_countries",
                   classes=dict(zip(NEED_CLASSES.values(), NEED_COLORS)))
plt.show()

In [ ]:
# Interactive version
m_need = geemap.Map(center=[3, 25], zoom=4)
for name, r in results.items():
    m_need.addLayer(r["need"].clip(r["study_area"]), need_vis, f"FLR need - {name}")
    m_need.addLayer(r["need_class"].clip(r["study_area"]), need_class_vis, f"Need classes - {name}", False)
    m_need.addLayer(outline(r["study_area"]), {"palette": ["black"]}, f"Outline - {name}")
m_need.add_colorbar(need_vis, label="FLR need index")
m_need.add_legend(title="Need class",
                  legend_dict=dict(zip(NEED_CLASSES.values(), NEED_COLORS)))
m_need

## 23. What is still needed for a top-tier journal

Sections 11-22 give the robustness checks reviewers expect of any multi-criteria map. On their own they are not enough for a Nature-family journal, where the usual bar is a new finding with validation, not a weighted overlay. The points below are ordered by how much they matter.

**Design issues to fix first (they affect every result)**

1. **Direction of each factor.** The original index adds all normalised factors with a positive sign, so high NDVI, high erosion and high biomass all push it the same way. Section 22 fixes this for a *degradation / need* reading, but the signs there are defaults: confirm each one and cite a source.
2. **Categorical layers are min-max scaled as if they were quantities.** Soil texture class, WorldCover code, GHSL settlement class, crop mask, fire mask and loss *year* have no meaningful order. Section 22 scores land cover per class and turns fire, cropland and settlement into present / absent; the land-cover scores are placeholders to replace with values from literature or experts, and soil texture still needs a per-class score before it can be used.
3. **The `soilTexture` weight.** It is corrected to 0.089 in this notebook; apply the same correction in the Code Editor script, and report the corrected weight table.
4. **AHP provenance.** Report the pairwise comparison matrix, who filled it in, and the consistency ratio (CR < 0.1). Twenty criteria in one matrix is beyond what AHP handles reliably; a two-level hierarchy (climate, soil, vegetation, human pressure) is easier to defend.
5. **Simplified sub-models.** `LS = 0.017 x slope`, `K = organic carbon / 10` and `C = 1 - NDVI` are placeholders. Use a published RUSLE parameterisation, or drop erosion and cite an existing product.
6. **Class thresholds.** With weights summing to 1, most land falls in one or two of the fixed 0.25-wide classes. Section 22 uses pooled quartiles; report the break values, and consider Jenks breaks as an alternative.
7. **Masked inputs.** The index only exists where every factor has data. Tree loss needed filling with 0 (done in section 3); check the null counts printed in section 11 for any other layer with gaps.
8. **Input vintage and resolution.** A single 2017 GRACE month at about 100 km, GPW 2000 population, 2010 biomass and GFSAD at 1 km are mixed with 10 m Sentinel-2. State the effective resolution of the result and prefer multi-year means over single months.

**Analyses that would make the paper**

9. **Validation with real restoration outcomes.** Field plots or project polygons where restoration succeeded and failed (national FLR/AFR100 programmes, Restor, TerraMatch, FAO SEPAL collections) are the strongest evidence. Section 20 is only a proxy.
10. **Comparison with published maps**: the WRI/IUCN Atlas of Forest Landscape Restoration Opportunities, Bastin et al. 2019 tree restoration potential, Griscom et al. 2017 reforestation potential, Strassburg et al. 2020 priority areas, and each country's AFR100 / Bonn Challenge pledge. Agreement and disagreement are both findings.
11. **Learn the weights from data.** Train a model (random forest, gradient boosting or MaxEnt) on observed natural regeneration and compare variable importance (SHAP) with the AHP weights; report spatially blocked cross-validation.
12. **Benefits and costs.** Carbon sequestration potential (Cook-Patton et al. 2020 accumulation rates), biodiversity value (range-size rarity, Key Biodiversity Areas), opportunity cost of land and people affected. A priority map optimised for several objectives is what recent high-impact restoration papers offer.
13. **Future climate.** Recompute the climate factors under CMIP6 scenarios (for example SSP2-4.5 and SSP5-8.5 at 2050) to show whether today's priority areas remain suitable.
14. **Spatial statistics.** Hotspot analysis (Getis-Ord Gi*) and spatial autocorrelation (Moran's I) of the index, and uncertainty maps from section 16 run wall-to-wall.
15. **Why these five countries?** Frame the selection (a gradient of biomes, pledges and governance) or extend to all AFR100 countries, since the workflow already runs per country.

**Reporting**

16. Share code and data (Zenodo DOI), give every dataset's version and citation, and report areas with their Monte Carlo intervals instead of single numbers.